# PyTorch Interview Challenge

This is the final capstone for AI Engineer, Machine Learning Engineer, and Generative AI Engineer interview preparation. It is a practical test—not a tutorial. It combines active recall, live coding, shape reasoning, debugging, and verbal explanation.

**Estimated time:** 4–6 hours. Sections also work independently for revision.

> Recognizing code is not enough. Can you write it, explain it, predict its shape, debug it, and use it under interview pressure?

The notebook contains well over 150 prompts, including 25 debugging cases, 20+ shape questions, two mock rounds, a live-coding task, and a broken-pipeline capstone.


## 1. Interview Rules

Try each problem before opening the solution.

For coding questions:

1. Predict the result first.
2. Write the code.
3. Run the validation.
4. Fix mistakes yourself.
5. Only then open the solution.

Complete your first attempt without internet or documentation. Starter cells use `None`; validators report **PENDING** until you replace it, so the pristine notebook can still execute top to bottom.

| Task | Practice target |
|---|---:|
| Syntax question | 1–2 minutes |
| Shape question | 30–60 seconds |
| Debugging question | 2–5 minutes |
| Model-building question | 5–10 minutes |
| Full training challenge | 15–30 minutes |

These are practice targets, not strict limits. Record both correctness and confidence.


## 2. Setup


In [ ]:
import math
import time

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    TensorDataset,
    DataLoader,
    random_split,
)

torch.manual_seed(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)


In [ ]:
def pending(name, value):
    # Keep starter notebooks executable while preserving a real assertion path.
    if value is None:
        print(f"PENDING: {name}")
        return True
    return False


def count_parameters(model, trainable_only=False):
    params = (p for p in model.parameters() if p.requires_grad) if trainable_only else model.parameters()
    return sum(p.numel() for p in params)


## 3. Tensor Warm-Up


### 15 recall prompts

Answer aloud or on paper before opening each answer.

**Q1. 🟢 Create `tensor([1, 2, 3, 4])`.**

<details>
<summary>Show answer</summary>

`torch.tensor([1, 2, 3, 4])`

</details>

**Q2. 🟢 Create a standard-normal tensor of shape `(32, 10)`.**

<details>
<summary>Show answer</summary>

`torch.randn(32, 10)`

</details>

**Q3. 🟢 Create integer values 0 through 9.**

<details>
<summary>Show answer</summary>

`torch.arange(10)`

</details>

**Q4. 🟢 Convert `x` to `float32`.**

<details>
<summary>Show answer</summary>

`x.float()` or `x.to(torch.float32)`

</details>

**Q5. 🟢 Find the index of the maximum value in a 1-D tensor.**

<details>
<summary>Show answer</summary>

`x.argmax()`

</details>

**Q6. 🟡 Select only positive values from `x`.**

<details>
<summary>Show answer</summary>

`x[x > 0]`

</details>

**Q7. 🟡 Replace negative values with zero without a loop.**

<details>
<summary>Show answer</summary>

`x.clamp_min(0)` or `torch.where(x < 0, 0, x)`

</details>

**Q8. 🟢 Create a `4 × 4` identity matrix.**

<details>
<summary>Show answer</summary>

`torch.eye(4)`

</details>

**Q9. 🟡 Create `0, .25, .5, .75, 1`.**

<details>
<summary>Show answer</summary>

`torch.linspace(0, 1, 5)`

</details>

**Q10. 🟡 Return the top three values and indices.**

<details>
<summary>Show answer</summary>

`values, indices = torch.topk(x, 3)`

</details>

**Q11. 🟢 Create zeros shaped like `x`.**

<details>
<summary>Show answer</summary>

`torch.zeros_like(x)`

</details>

**Q12. 🟡 Test elementwise finiteness.**

<details>
<summary>Show answer</summary>

`torch.isfinite(x)`

</details>

**Q13. 🟡 Count values greater than zero.**

<details>
<summary>Show answer</summary>

`(x > 0).sum()`

</details>

**Q14. 🟡 Convert a scalar tensor to a Python number.**

<details>
<summary>Show answer</summary>

`x.item()`

</details>

**Q15. 🟡 Reproduce `x` without sharing storage.**

<details>
<summary>Show answer</summary>

`x.clone()`

</details>


### Warm-up validation sprint

Fill all five expressions without documentation.

Predict first, then complete the next cell. Run its validation before opening the solution.


In [ ]:
wu_1 = None  # [1, 2, 3, 4]
wu_2 = None  # random normal, (32, 10)
wu_3 = None  # integers 0..9
_wu_x = torch.tensor([-2.0, 1.0, -1.0, 3.0])
wu_4 = None  # positive values only
wu_5 = None  # negatives replaced by zero


In [ ]:
_wu_answers = [wu_1, wu_2, wu_3, wu_4, wu_5]
if any(v is None for v in _wu_answers):
    print("PENDING: warm-up sprint")
else:
    assert torch.equal(wu_1, torch.tensor([1, 2, 3, 4]))
    assert wu_2.shape == (32, 10)
    assert torch.equal(wu_3, torch.arange(10))
    assert torch.equal(wu_4, torch.tensor([1.0, 3.0]))
    assert torch.equal(wu_5, torch.tensor([0.0, 1.0, 0.0, 3.0]))
    print("PASS: warm-up sprint")


<details>
    <summary>Show solution</summary>

    ```python
    wu_1 = torch.tensor([1, 2, 3, 4])
wu_2 = torch.randn(32, 10)
wu_3 = torch.arange(10)
wu_4 = _wu_x[_wu_x > 0]
wu_5 = _wu_x.clamp_min(0)
    ```

    </details>

**Interview takeaway:** Boolean masks and factory functions should be automatic under time pressure.


## 4. Shape Reasoning Challenge


### 25 core shape predictions

Answer aloud or on paper before opening each answer.

**S1. `x=(32,3,224,224); x.flatten(1)`**

<details>
<summary>Show answer</summary>

`(32, 150528)`

</details>

**S2. `x=(16,20,128); x.mean(dim=1)`**

<details>
<summary>Show answer</summary>

`(16, 128)`

</details>

**S3. `x=(8,12,64,64); x.mean(dim=(2,3))`**

<details>
<summary>Show answer</summary>

`(8, 12)`

</details>

**S4. `x=(32,10); x.argmax(dim=1, keepdim=True)`**

<details>
<summary>Show answer</summary>

`(32, 1)`

</details>

**S5. `x=(2,3,4); x.reshape(6,4)`**

<details>
<summary>Show answer</summary>

`(6, 4)`

</details>

**S6. `x=(7,1,9); x.squeeze(1)`**

<details>
<summary>Show answer</summary>

`(7, 9)`

</details>

**S7. `x=(7,1,9); x.squeeze()`**

<details>
<summary>Show answer</summary>

`(7, 9)`

</details>

**S8. `x=(5,6); x.unsqueeze(0)`**

<details>
<summary>Show answer</summary>

`(1, 5, 6)`

</details>

**S9. `x=(5,6); x.unsqueeze(-1)`**

<details>
<summary>Show answer</summary>

`(5, 6, 1)`

</details>

**S10. `x=(2,3,4); x.transpose(1,2)`**

<details>
<summary>Show answer</summary>

`(2, 4, 3)`

</details>

**S11. `x=(8,3,32,32); x.permute(0,2,3,1)`**

<details>
<summary>Show answer</summary>

`(8, 32, 32, 3)`

</details>

**S12. `a,b=(3,4); torch.cat([a,b], dim=0)`**

<details>
<summary>Show answer</summary>

`(6, 4)`

</details>

**S13. `a,b=(3,4); torch.cat([a,b], dim=1)`**

<details>
<summary>Show answer</summary>

`(3, 8)`

</details>

**S14. `a,b=(4,5); torch.stack([a,b], dim=1)`**

<details>
<summary>Show answer</summary>

`(4, 2, 5)`

</details>

**S15. `x=(10,20); nn.Linear(20,7)(x)`**

<details>
<summary>Show answer</summary>

`(10, 7)`

</details>

**S16. `ids=(8,12); nn.Embedding(1000,64)(ids)`**

<details>
<summary>Show answer</summary>

`(8, 12, 64)`

</details>

**S17. `Conv2d(3,16,3,padding=1)` on `(8,3,32,32)`**

<details>
<summary>Show answer</summary>

`(8, 16, 32, 32)`

</details>

**S18. `Conv2d(3,16,3,stride=2,padding=1)` on `(8,3,32,32)`**

<details>
<summary>Show answer</summary>

`(8, 16, 16, 16)`

</details>

**S19. `x=(4,5,6); x.sum(dim=-1)`**

<details>
<summary>Show answer</summary>

`(4, 5)`

</details>

**S20. `A=(3,4); B=(4,2); A @ B`**

<details>
<summary>Show answer</summary>

`(3, 2)`

</details>

**S21. `A=(8,3,4); B=(4,2); A @ B`**

<details>
<summary>Show answer</summary>

`(8, 3, 2)`

</details>

**S22. `x=(8,128,768), H=12`; split heads**

<details>
<summary>Show answer</summary>

`(8, 12, 128, 64)`

</details>

**S23. `attn=(8,12,128,128), V=(8,12,128,64); attn @ V`**

<details>
<summary>Show answer</summary>

`(8, 12, 128, 64)`

</details>

**S24. LM hidden `(8,128,768)` projected to vocabulary 50,000**

<details>
<summary>Show answer</summary>

`(8, 128, 50000)`

</details>

**S25. `x=(2,3,4); x.flatten(0,1)`**

<details>
<summary>Show answer</summary>

`(6, 4)`

</details>


### Shape traps

Answer aloud or on paper before opening each answer.

**Trap1. Why can `view()` fail after `permute()`?**

<details>
<summary>Show answer</summary>

The tensor may be non-contiguous. Use `reshape()` or `.contiguous().view(...)`.

</details>

**Trap2. `x=(1,3,1); x.squeeze(0)`**

<details>
<summary>Show answer</summary>

`(3, 1)`—only dimension 0 is removed.

</details>

**Trap3. `x=(1,3,1); x.squeeze()`**

<details>
<summary>Show answer</summary>

`(3,)`—all singleton dimensions are removed.

</details>

**Trap4. Can tensors `(2,3)` and `(2,4)` be concatenated on dim 1?**

<details>
<summary>Show answer</summary>

Yes: `(2, 7)`; non-concatenated dimensions match.

</details>

**Trap5. Can tensors `(2,3)` and `(2,4)` be stacked?**

<details>
<summary>Show answer</summary>

No; every input shape must match.

</details>

**Trap6. `torch.max(x, dim=1)` returns what?**

<details>
<summary>Show answer</summary>

A named tuple of values and indices, each with dim 1 removed unless `keepdim=True`.

</details>

**Trap7. `Linear(4,5)` on `(2,3,4)`**

<details>
<summary>Show answer</summary>

`(2, 3, 5)`; Linear transforms the last dimension.

</details>

**Trap8. `Embedding(10,6)` on scalar token ID**

<details>
<summary>Show answer</summary>

`(6,)`.

</details>

**Trap9. `x=(4,5); x.T.unsqueeze(0)`**

<details>
<summary>Show answer</summary>

`(1, 5, 4)`.

</details>

**Trap10. `x=(2,3,4); x[0]`**

<details>
<summary>Show answer</summary>

`(3, 4)`; integer indexing removes a dimension.

</details>


## 5. Tensor Manipulation Challenge


### Manipulation prompts

Answer aloud or on paper before opening each answer.

**M1. Flatten `(32,3,28,28)` without hard-coding batch size.**

<details>
<summary>Show answer</summary>

`x.flatten(1)` or `x.reshape(x.size(0), -1)`.

</details>

**M2. Convert NCHW to NHWC.**

<details>
<summary>Show answer</summary>

`x.permute(0, 2, 3, 1)`.

</details>

**M3. Select each sample's labeled class score without a loop.**

<details>
<summary>Show answer</summary>

`scores[torch.arange(scores.size(0)), labels]` or `scores.gather(1, labels[:,None]).squeeze(1)`.

</details>

**M4. Normalize each feature column independently.**

<details>
<summary>Show answer</summary>

`(x-x.mean(0,keepdim=True))/x.std(0,keepdim=True).clamp_min(1e-8)`.

</details>

**M5. Return top-5 class indices per sample.**

<details>
<summary>Show answer</summary>

`logits.topk(5, dim=1).indices`.

</details>

**M6. Reverse a sequence dimension 1.**

<details>
<summary>Show answer</summary>

`x.flip(1)`.

</details>

**M7. Repeat each row three times as a new dimension.**

<details>
<summary>Show answer</summary>

`x.unsqueeze(1).expand(-1, 3, -1)` (view) or `.repeat(1,3,1)` (copy).

</details>

**M8. Create a pairwise equality matrix for labels `(B,)`.**

<details>
<summary>Show answer</summary>

`labels[:, None] == labels[None, :]`.

</details>

**M9. Replace NaN and infinities with finite defaults.**

<details>
<summary>Show answer</summary>

`torch.nan_to_num(x)`.

</details>

**M10. One-hot encode class IDs into C columns.**

<details>
<summary>Show answer</summary>

`F.one_hot(labels, num_classes=C)`.

</details>


## 6. Broadcasting Challenge


### 15 broadcast decisions

Answer aloud or on paper before opening each answer.

**B1. Will `(32,10) + (10,)` broadcast? Output?**

<details>
<summary>Show answer</summary>

Yes, `(32, 10)`.

</details>

**B2. Will `(8,1,128) + (1,20,1)` broadcast? Output?**

<details>
<summary>Show answer</summary>

Yes, `(8, 20, 128)`.

</details>

**B3. Will `(3,4) + (3,)` broadcast?**

<details>
<summary>Show answer</summary>

No: trailing dimensions 4 and 3 conflict.

</details>

**B4. Will `(5,1) * (7,)` broadcast? Output?**

<details>
<summary>Show answer</summary>

Yes, `(5, 7)`.

</details>

**B5. Will `(2,3,4) + (3,1)` broadcast? Output?**

<details>
<summary>Show answer</summary>

Yes, `(2, 3, 4)`.

</details>

**B6. Will `(2,3,4) + (2,1)` broadcast?**

<details>
<summary>Show answer</summary>

No: aligned dimensions 3 and 2 conflict.

</details>

**B7. Will `(1,) + (4,5,6)` broadcast? Output?**

<details>
<summary>Show answer</summary>

Yes, `(4, 5, 6)`.

</details>

**B8. Will scalar `()` multiply `(8,9)`?**

<details>
<summary>Show answer</summary>

Yes, `(8, 9)`.

</details>

**B9. Will `(4,1,8,1) + (3,1,7)` broadcast? Output?**

<details>
<summary>Show answer</summary>

Yes, `(4, 3, 8, 7)`.

</details>

**B10. Will `(10,5) + (10,1)` broadcast? Output?**

<details>
<summary>Show answer</summary>

Yes, `(10, 5)`.

</details>

**B11. Will `(10,5) + (1,10)` broadcast?**

<details>
<summary>Show answer</summary>

No: 5 conflicts with 10.

</details>

**B12. Why does subtracting `x.mean(dim=0, keepdim=True)` from `x=(64,128)` work?**

<details>
<summary>Show answer</summary>

The mean is `(1,128)`; its singleton batch dimension expands to 64.

</details>

**B13. For per-channel NCHW scaling, what shape should scale have?**

<details>
<summary>Show answer</summary>

`(1, C, 1, 1)`.

</details>

**B14. What does `expand` do differently from `repeat`?**

<details>
<summary>Show answer</summary>

`expand` returns a stride-0 view when possible; `repeat` copies data.

</details>

**B15. How can you verify broadcastability without performing arithmetic?**

<details>
<summary>Show answer</summary>

Use `torch.broadcast_shapes(shape_a, shape_b)`.

</details>


## 7. Matrix Multiplication Challenge


### 12 matrix questions

Answer aloud or on paper before opening each answer.

**MM1. `(32,128) @ (128,64)`**

<details>
<summary>Show answer</summary>

`(32, 64)`.

</details>

**MM2. `(16,20,64)` bmm `(16,64,32)`**

<details>
<summary>Show answer</summary>

`(16, 20, 32)`.

</details>

**MM3. Q,K `(8,12,128,64)`; `Q @ K.transpose(-2,-1)`**

<details>
<summary>Show answer</summary>

`(8, 12, 128, 128)`.

</details>

**MM4. `(4,) @ (4,)`**

<details>
<summary>Show answer</summary>

Scalar `()` dot product.

</details>

**MM5. `(3,4) @ (4,)`**

<details>
<summary>Show answer</summary>

`(3,)`.

</details>

**MM6. `(4,) @ (4,5)`**

<details>
<summary>Show answer</summary>

`(5,)`.

</details>

**MM7. `(2,3,4) @ (1,4,6)`**

<details>
<summary>Show answer</summary>

`(2, 3, 6)`; batch dimensions broadcast.

</details>

**MM8. Can `torch.bmm` broadcast batch dimensions?**

<details>
<summary>Show answer</summary>

No; both inputs must be 3-D with the same batch size.

</details>

**MM9. Why is elementwise `*` not a Linear layer?**

<details>
<summary>Show answer</summary>

It does not sum over the shared feature dimension; matrix multiplication does.

</details>

**MM10. Weight stored by `Linear(in=4,out=7)` has what shape?**

<details>
<summary>Show answer</summary>

`(7, 4)`; forward effectively computes `x @ weight.T + bias`.

</details>

**MM11. Attention scores `(B,H,T,T)` times V `(B,H,T,Dh)`**

<details>
<summary>Show answer</summary>

`(B, H, T, Dh)`.

</details>

**MM12. What condition must inner matrix dimensions satisfy?**

<details>
<summary>Show answer</summary>

Left last dimension equals right second-to-last dimension.

</details>


## 8. Vectorization Challenge


### 10 remove-the-loop prompts

Answer aloud or on paper before opening each answer.

**V1. Replace loop appending positive values.**

<details>
<summary>Show answer</summary>

`x[x > 0]`.

</details>

**V2. Replace loop squaring each element.**

<details>
<summary>Show answer</summary>

`x ** 2`.

</details>

**V3. Replace per-row prediction loop.**

<details>
<summary>Show answer</summary>

`logits.argmax(dim=1)`.

</details>

**V4. Replace loop computing row sums.**

<details>
<summary>Show answer</summary>

`x.sum(dim=1)`.

</details>

**V5. Replace nested loop computing pairwise distances.**

<details>
<summary>Show answer</summary>

`torch.cdist(x, y)` or broadcast subtraction followed by norm.

</details>

**V6. Replace loop selecting labeled logits.**

<details>
<summary>Show answer</summary>

Advanced indexing or `gather`.

</details>

**V7. Replace loop clipping values to `[0,1]`.**

<details>
<summary>Show answer</summary>

`x.clamp(0, 1)`.

</details>

**V8. Replace loop counting correct predictions.**

<details>
<summary>Show answer</summary>

`(pred == y).sum()`.

</details>

**V9. Replace loop standardizing columns.**

<details>
<summary>Show answer</summary>

Column reductions with `keepdim=True`, then broadcast.

</details>

**V10. Replace loop applying one model to a batch sample-by-sample.**

<details>
<summary>Show answer</summary>

Pass the entire batch to the model.

</details>


## 9. Autograd Challenge


### 15 Autograd prompts

Answer aloud or on paper before opening each answer.

**A1. `x=3`, `y=2x²+5x`: predict `dy/dx`.**

<details>
<summary>Show answer</summary>

`4x+5 = 17`.

</details>

**A2. `x=2`, `a=4x`, `y=a³`: predict `dy/dx`.**

<details>
<summary>Show answer</summary>

`3a²·4 = 768`.

</details>

**A3. After `y=x²; y.backward(); y.backward()` at x=3, final grad?**

<details>
<summary>Show answer</summary>

`12`; gradients accumulate (with retained/rebuilt graph as needed).

</details>

**A4. Does `x.detach()` share storage and track the original graph?**

<details>
<summary>Show answer</summary>

It usually shares storage but is detached from the graph.

</details>

**A5. Does an operation inside `torch.no_grad()` produce grad history?**

<details>
<summary>Show answer</summary>

No, unless using factory-function exceptions such as creating a Parameter.

</details>

**A6. Why must non-scalar outputs receive a gradient argument to `backward`?**

<details>
<summary>Show answer</summary>

Autograd needs the vector-Jacobian product seed.

</details>

**A7. What is a leaf tensor?**

<details>
<summary>Show answer</summary>

A user-created grad-requiring tensor not produced by a tracked operation; parameter gradients accumulate on leaves.

</details>

**A8. When is `.grad` populated by default?**

<details>
<summary>Show answer</summary>

For leaf tensors with `requires_grad=True`; call `retain_grad()` for non-leaves.

</details>

**A9. What does `optimizer.zero_grad(set_to_none=True)` change?**

<details>
<summary>Show answer</summary>

Sets grads to `None`, often avoiding a fill and making missing gradients distinguishable from zero.

</details>

**A10. Why avoid `.data` for updates?**

<details>
<summary>Show answer</summary>

It bypasses Autograd safety/version tracking; use `no_grad()` for manual updates.

</details>

**A11. What creates a graph: module construction or forward execution?**

<details>
<summary>Show answer</summary>

Forward execution dynamically records operations.

</details>

**A12. If `z=x.detach()*2`, can a loss through z update x?**

<details>
<summary>Show answer</summary>

No.

</details>

**A13. How inspect gradient norm?**

<details>
<summary>Show answer</summary>

After backward: `p.grad.norm()` for a parameter with non-None grad.

</details>

**A14. What does `loss.item()` return?**

<details>
<summary>Show answer</summary>

A detached Python scalar; calling it can synchronize an accelerator.

</details>

**A15. Why does a second backward through the same graph often fail?**

<details>
<summary>Show answer</summary>

Saved intermediates were freed; use `retain_graph=True` only when genuinely needed.

</details>


### Manual gradient prediction

Set `predicted_grad` before running Autograd.

Predict first, then complete the next cell. Run its validation before opening the solution.


In [ ]:
predicted_grad = None
ag_x = torch.tensor(3.0, requires_grad=True)
ag_y = 2 * ag_x**2 + 5 * ag_x


In [ ]:
if not pending("manual gradient", predicted_grad):
    ag_y.backward()
    assert predicted_grad == 17
    assert ag_x.grad.item() == 17.0
    print("PASS: prediction matches Autograd")


<details>
    <summary>Show solution</summary>

    ```python
    predicted_grad = 17
    ```

    </details>

**Interview takeaway:** Differentiate first; use Autograd to validate your reasoning, not replace it.


### Explain the bug — Autograd


### Three bug diagnoses

Answer aloud or on paper before opening each answer.

**Bug1. `x=torch.tensor(2.0); y=x**2; y.backward()`**

<details>
<summary>Show answer</summary>

`x` does not require gradients. Create it with `requires_grad=True`.

</details>

**Bug2. `loss.backward(); optimizer.zero_grad(); optimizer.step()`**

<details>
<summary>Show answer</summary>

The computed gradients are erased before the step. Zero before forward/backward.

</details>

**Bug3. `features=backbone(X).detach(); logits=head(features)`**

<details>
<summary>Show answer</summary>

Detach cuts the graph, so the backbone receives no gradients.

</details>


## 10. Neural Network Construction


### 12 architecture decisions

Answer aloud or on paper before opening each answer.

**N1. Build `10→32→ReLU→2`; expected `(8,2)`.**

<details>
<summary>Show answer</summary>

`nn.Sequential(nn.Linear(10,32), nn.ReLU(), nn.Linear(32,2))`.

</details>

**N2. Build `20→64→ReLU→Dropout(.3)→32→GELU→5`.**

<details>
<summary>Show answer</summary>

Use two hidden Linear layers, the named activations, Dropout, then `Linear(32,5)`.

</details>

**N3. Binary classifier with 100 features: final layer and activation?**

<details>
<summary>Show answer</summary>

`Linear(...,1)` and no in-model sigmoid when using `BCEWithLogitsLoss`.

</details>

**N4. Embedding classifier steps?**

<details>
<summary>Show answer</summary>

Token IDs → Embedding → mask-aware/ordinary mean over tokens → Linear logits.

</details>

**N5. Two-block CIFAR CNN output?**

<details>
<summary>Show answer</summary>

Two Conv/ReLU/Pool blocks, flatten, then Linear to 10 logits; calculate the flattened size.

</details>

**N6. Where should learnable layers be created?**

<details>
<summary>Show answer</summary>

In `__init__`, so parameters are registered and reused.

</details>

**N7. Why not create `nn.Linear` inside `forward` each call?**

<details>
<summary>Show answer</summary>

It reinitializes and may leave parameters unregistered/outside the optimizer.

</details>

**N8. When is `nn.Sequential` insufficient?**

<details>
<summary>Show answer</summary>

Branching, residuals, multiple inputs/outputs, masking, or data-dependent control flow.

</details>

**N9. Sequential for `Linear→ReLU→Linear`?**

<details>
<summary>Show answer</summary>

Yes.

</details>

**N10. Sequential for mask-aware mean pooling?**

<details>
<summary>Show answer</summary>

A custom module is clearer.

</details>

**N11. Sequential for `x + block(x)`?**

<details>
<summary>Show answer</summary>

Use a custom module for the residual connection.

</details>

**N12. Sequential for two independent towers joined later?**

<details>
<summary>Show answer</summary>

Use a custom module.

</details>


### `nn.Module` from memory

Implement `10 → 32 → ReLU → 2` with `__init__`, `super()`, registered layers, and `forward`.

Predict first, then complete the next cell. Run its validation before opening the solution.


In [ ]:
class MyModel(nn.Module):
    pass

my_model = None


In [ ]:
if my_model is None:
    print("PENDING: custom nn.Module")
else:
    assert isinstance(my_model, nn.Module)
    assert my_model(torch.randn(8, 10)).shape == (8, 2)
    assert count_parameters(my_model) == 418
    print("PASS: custom module")


<details>
    <summary>Show solution</summary>

    ```python
    class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

my_model = MyModel()
    ```

    </details>

**Interview takeaway:** Submodules assigned as attributes are registered automatically.


## 11. Parameter Counting


### 15 parameter questions

Answer aloud or on paper before opening each answer.

**P1. `Linear(128,64)`**

<details>
<summary>Show answer</summary>

`128×64+64 = 8,256`.

</details>

**P2. `Embedding(50000,768)`**

<details>
<summary>Show answer</summary>

`38,400,000`.

</details>

**P3. `Conv2d(3,16,3)` with bias**

<details>
<summary>Show answer</summary>

`16×3×3×3+16 = 448`.

</details>

**P4. `LayerNorm(768)` with affine parameters**

<details>
<summary>Show answer</summary>

`768` scales + `768` biases = `1,536`.

</details>

**P5. `Linear(10,32,bias=False)`**

<details>
<summary>Show answer</summary>

`320`.

</details>

**P6. MLP `10→32→2` with biases**

<details>
<summary>Show answer</summary>

`10×32+32 + 32×2+2 = 418`.

</details>

**P7. `Embedding(100,16)` followed by `Linear(16,3)`**

<details>
<summary>Show answer</summary>

`1,600 + 48 + 3 = 1,651`.

</details>

**P8. `Conv2d(16,32,3,groups=16)`**

<details>
<summary>Show answer</summary>

`32×(16/16)×3×3 + 32 = 320`.

</details>

**P9. Does ReLU add parameters?**

<details>
<summary>Show answer</summary>

No.

</details>

**P10. Does Dropout add parameters?**

<details>
<summary>Show answer</summary>

No.

</details>

**P11. Does BatchNorm1d(64) have trainable parameters?**

<details>
<summary>Show answer</summary>

Yes: 64 scales and 64 biases; running stats are buffers, not parameters.

</details>

**P12. How count all model parameters?**

<details>
<summary>Show answer</summary>

`sum(p.numel() for p in model.parameters())`.

</details>

**P13. How count trainable-only parameters?**

<details>
<summary>Show answer</summary>

Filter with `if p.requires_grad`.

</details>

**P14. Does freezing remove parameters from `model.parameters()`?**

<details>
<summary>Show answer</summary>

No; it changes `requires_grad`, so explicitly filter if desired.

</details>

**P15. Why can parameter count differ from optimizer state size?**

<details>
<summary>Show answer</summary>

Optimizers may store extra tensors per optimized parameter, such as Adam moments.

</details>


## 12. Loss Function Selection — 🔥 Interview Essential


### 20 output/target/loss decisions

Answer aloud or on paper before opening each answer.

**L1. House-price regression**

<details>
<summary>Show answer</summary>

Output `(B,1)`; target `(B,1)` float; `MSELoss` (or L1); inference raw value.

</details>

**L2. Single-label binary classification**

<details>
<summary>Show answer</summary>

Output `(B,1)` raw logits; target `(B,1)` float; `BCEWithLogitsLoss`; sigmoid + threshold.

</details>

**L3. C-class single-label classification**

<details>
<summary>Show answer</summary>

Output `(B,C)` logits; target `(B,)` long; `CrossEntropyLoss`; argmax.

</details>

**L4. C-label multilabel classification**

<details>
<summary>Show answer</summary>

Output/target `(B,C)`, target float; `BCEWithLogitsLoss`; per-label sigmoid + thresholds.

</details>

**L5. 3-class image segmentation**

<details>
<summary>Show answer</summary>

Output `(B,3,H,W)`; target `(B,H,W)` long; `CrossEntropyLoss`; argmax over dim 1.

</details>

**L6. Binary segmentation**

<details>
<summary>Show answer</summary>

Output/target `(B,1,H,W)` floats for target; `BCEWithLogitsLoss`; sigmoid threshold.

</details>

**L7. Next-token language modeling**

<details>
<summary>Show answer</summary>

Logits `(B,T,V)`, targets `(B,T)` long; flatten for `CrossEntropyLoss`; token argmax/sampling.

</details>

**L8. Robust scalar regression with outliers**

<details>
<summary>Show answer</summary>

Same float shapes; `L1Loss` or SmoothL1; raw value.

</details>

**L9. Probability distribution target over classes**

<details>
<summary>Show answer</summary>

`CrossEntropyLoss` supports same-shape floating probability targets; alternatively KL with log-probs, depending objective.

</details>

**L10. Ordinal categories treated as ordinary exclusive classes**

<details>
<summary>Show answer</summary>

`(B,C)` logits, `(B,)` long, CrossEntropy; note that it ignores ordinal distance.

</details>

**L11. Multi-output regression with 5 targets**

<details>
<summary>Show answer</summary>

Output/target `(B,5)` float; MSE/L1; raw vector.

</details>

**L12. Class-weighted multiclass classification**

<details>
<summary>Show answer</summary>

CrossEntropy with a length-C weight tensor on the same device.

</details>

**L13. Imbalanced binary task needing positive weighting**

<details>
<summary>Show answer</summary>

BCEWithLogitsLoss with `pos_weight`; raw logits.

</details>

**L14. Contrastive similarity over candidate IDs**

<details>
<summary>Show answer</summary>

Often CrossEntropy over a similarity-logit matrix and long correct indices.

</details>

**L15. Autoencoder pixel reconstruction**

<details>
<summary>Show answer</summary>

Output and target same floating shape; MSE or task-appropriate reconstruction loss.

</details>

**L16. Predict one of 50,000 vocabulary tokens**

<details>
<summary>Show answer</summary>

50,000 logits per position and long token IDs; CrossEntropy.

</details>

**L17. Why raw logits for CrossEntropy?**

<details>
<summary>Show answer</summary>

It combines log-softmax and NLL stably.

</details>

**L18. Why raw logits for BCEWithLogits?**

<details>
<summary>Show answer</summary>

It combines sigmoid and BCE stably.

</details>

**L19. What target dtype does CrossEntropy expect for class indices?**

<details>
<summary>Show answer</summary>

`torch.long`.

</details>

**L20. What target dtype does BCEWithLogits expect?**

<details>
<summary>Show answer</summary>

Floating point, same shape as logits.

</details>


### Loss bug hunt

Answer aloud or on paper before opening each answer.

**Loss bug1. Softmax then CrossEntropyLoss**

<details>
<summary>Show answer</summary>

Remove softmax; pass logits directly.

</details>

**Loss bug2. Sigmoid then BCEWithLogitsLoss**

<details>
<summary>Show answer</summary>

Remove sigmoid; pass logits directly.

</details>

**Loss bug3. Float class-index targets with CrossEntropyLoss**

<details>
<summary>Show answer</summary>

Use long targets of shape `(B,)`.

</details>


## 13. Optimizer Questions


### Optimizer recall

Answer aloud or on paper before opening each answer.

**O1. Create AdamW, lr `3e-4`, decay `.01`.**

<details>
<summary>Show answer</summary>

`optim.AdamW(model.parameters(), lr=3e-4, weight_decay=.01)`.

</details>

**O2. Create SGD, lr `.01`, momentum `.9`.**

<details>
<summary>Show answer</summary>

`optim.SGD(model.parameters(), lr=.01, momentum=.9)`.

</details>

**O3. Freeze backbone and optimize classifier only.**

<details>
<summary>Show answer</summary>

Set backbone `requires_grad=False`; pass classifier parameters or filtered trainable parameters.

</details>

**O4. Why AdamW rather than Adam with naive L2 coupling?**

<details>
<summary>Show answer</summary>

AdamW decouples weight decay from adaptive gradient updates.

</details>

**O5. When should an LR scheduler usually step?**

<details>
<summary>Show answer</summary>

Depends on scheduler: often after each optimizer step or epoch; metric schedulers step after validation. Read its contract.

</details>


## 14. Training Loop Challenge


### Training loop from memory

Write `train_one_epoch(model, loader, loss_fn, optimizer, device)` returning sample-weighted loss and accuracy. Include mode, movement, zeroing, forward, backward, step, and metrics. Do not use the solution until your implementation learns the supplied synthetic data.

Predict first, then complete the next cell. Run its validation before opening the solution.


In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    # YOUR CODE
    return None


In [ ]:
_train_X = torch.randn(256, 8)
_train_y = (_train_X[:, 0] + 0.7 * _train_X[:, 1] > 0).long()
_train_loader = DataLoader(TensorDataset(_train_X, _train_y), batch_size=32, shuffle=True)
if train_one_epoch(nn.Linear(8, 2), _train_loader, nn.CrossEntropyLoss(), optim.SGD(nn.Linear(8, 2).parameters(), lr=.1), torch.device("cpu")) is None:
    print("PENDING: training loop (return loss, accuracy when implemented)")
else:
    _tm = nn.Linear(8, 2).to(device)
    _to = optim.SGD(_tm.parameters(), lr=0.4)
    _before = nn.CrossEntropyLoss()(_tm(_train_X.to(device)), _train_y.to(device)).item()
    for _ in range(4):
        _metrics = train_one_epoch(_tm, _train_loader, nn.CrossEntropyLoss(), _to, device)
    _after = nn.CrossEntropyLoss()(_tm(_train_X.to(device)), _train_y.to(device)).item()
    assert len(_metrics) == 2 and all(isinstance(v, float) for v in _metrics)
    assert 0 <= _metrics[1] <= 1 and _after < _before
    print("PASS: training loop learns")


<details>
    <summary>Show solution</summary>

    ```python
    def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss = total_correct = total_samples = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(X)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        n = y.size(0)
        total_loss += loss.item() * n
        total_correct += (logits.argmax(1) == y).sum().item()
        total_samples += n
    return total_loss / total_samples, total_correct / total_samples
    ```

    </details>

**Interview takeaway:** The safe order is mode → batch/device → zero → forward → loss → backward → step → detached metrics.


## 15. Validation Loop Challenge


### Validation from memory

Write `evaluate` returning sample-weighted loss and accuracy. It must leave parameters unchanged.

Predict first, then complete the next cell. Run its validation before opening the solution.


In [ ]:
def evaluate(model, loader, loss_fn, device):
    # YOUR CODE
    return None


In [ ]:
_eval_model = nn.Linear(8, 2).to(device)
_snapshot = [p.detach().clone() for p in _eval_model.parameters()]
_eval_result = evaluate(_eval_model, _train_loader, nn.CrossEntropyLoss(), device)
if _eval_result is None:
    print("PENDING: validation loop")
else:
    assert len(_eval_result) == 2
    assert all(torch.equal(a, b) for a, b in zip(_snapshot, _eval_model.parameters()))
    assert not _eval_model.training and 0 <= _eval_result[1] <= 1
    print("PASS: validation is side-effect free")


<details>
    <summary>Show solution</summary>

    ```python
    def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = total_correct = total_samples = 0
    with torch.inference_mode():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss = loss_fn(logits, y)
            n = y.size(0)
            total_loss += loss.item() * n
            total_correct += (logits.argmax(1) == y).sum().item()
            total_samples += n
    return total_loss / total_samples, total_correct / total_samples
    ```

    </details>

**Interview takeaway:** `eval()` changes layer behavior; inference mode disables graph recording. They solve different problems.


### Train vs eval

Answer aloud or on paper before opening each answer.

**TE1. Does `model.eval()` disable gradients?**

<details>
<summary>Show answer</summary>

No.

</details>

**TE2. Does `no_grad()` turn off Dropout?**

<details>
<summary>Show answer</summary>

No.

</details>

**TE3. What should validation normally use?**

<details>
<summary>Show answer</summary>

Both `model.eval()` and `no_grad()`/`inference_mode()`.

</details>

**TE4. What does `train()` affect?**

<details>
<summary>Show answer</summary>

Modules such as Dropout and BatchNorm; it does not enable Autograd globally.

</details>

**TE5. Can you train while model is in eval mode?**

<details>
<summary>Show answer</summary>

Gradients can still compute, but Dropout/BatchNorm behavior is wrong for normal training.

</details>

**TE6. Does `eval()` freeze parameters?**

<details>
<summary>Show answer</summary>

No.

</details>

**TE7. Can `inference_mode()` be faster than `no_grad()`?**

<details>
<summary>Show answer</summary>

Often yes; it disables additional Autograd bookkeeping and is more restrictive.

</details>

**TE8. Why return to train mode after validation?**

<details>
<summary>Show answer</summary>

Otherwise stateful/stochastic training layers remain in evaluation behavior.

</details>

**TE9. Should validation call `optimizer.step()`?**

<details>
<summary>Show answer</summary>

No.

</details>

**TE10. Should validation shuffle?**

<details>
<summary>Show answer</summary>

Usually unnecessary; metrics should not depend on order.

</details>


### Loss aggregation

Answer aloud or on paper before opening each answer.

**Agg1. Why is summing batch means then dividing by dataset size wrong?**

<details>
<summary>Show answer</summary>

Batch means are already divided by batch size. Multiply each by its batch size, sum, and divide by total samples—especially when the final batch is smaller.

</details>


## 16. Dataset and DataLoader Challenge


### Custom Dataset from memory

Implement exact indexed feature/target retrieval.

Predict first, then complete the next cell. Run its validation before opening the solution.


In [ ]:
class CustomDataset(Dataset):
    pass

custom_ds = None


In [ ]:
_cd_X, _cd_y = torch.randn(11, 4), torch.arange(11)
if custom_ds is None:
    print("PENDING: CustomDataset")
else:
    assert len(custom_ds) == 11
    assert torch.equal(custom_ds[7][0], _cd_X[7])
    assert torch.equal(custom_ds[7][1], _cd_y[7])
    print("PASS: CustomDataset")


<details>
    <summary>Show solution</summary>

    ```python
    class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

custom_ds = CustomDataset(_cd_X, _cd_y)
    ```

    </details>


### DataLoader decisions

Answer aloud or on paper before opening each answer.

**DL1. Training loader settings?**

<details>
<summary>Show answer</summary>

`batch_size=32, shuffle=True`.

</details>

**DL2. Validation loader settings?**

<details>
<summary>Show answer</summary>

`batch_size=64, shuffle=False`.

</details>

**DL3. What happens to a partial final batch?**

<details>
<summary>Show answer</summary>

It is returned by default; `drop_last=True` discards it.

</details>

**DL4. What does `num_workers` do?**

<details>
<summary>Show answer</summary>

Uses subprocesses to load/transform batches in parallel; best value is workload/platform dependent.

</details>

**DL5. What does `pin_memory` do?**

<details>
<summary>Show answer</summary>

Places CPU batches in page-locked memory, which can speed host-to-CUDA copies with non-blocking transfer.

</details>


### Variable-length `collate_fn`

Return padded `input_ids` and a boolean attention mask for sequences `[1,2,3]`, `[4,5]`, `[6,7,8,9]`.

Predict first, then complete the next cell. Run its validation before opening the solution.


In [ ]:
def pad_collate(batch):
    # YOUR CODE
    return None

_sequences = [torch.tensor([1, 2, 3]), torch.tensor([4, 5]), torch.tensor([6, 7, 8, 9])]


In [ ]:
_padded_result = pad_collate(_sequences)
if _padded_result is None:
    print("PENDING: padding collate_fn")
else:
    _ids, _mask = _padded_result
    assert _ids.shape == _mask.shape == (3, 4)
    assert _ids[1, 2:].eq(0).all() and not _mask[1, 2:].any()
    assert _mask.sum().item() == 9
    print("PASS: padding and mask")


<details>
    <summary>Show solution</summary>

    ```python
    def pad_collate(batch):
    max_len = max(len(x) for x in batch)
    ids = torch.zeros(len(batch), max_len, dtype=torch.long)
    mask = torch.zeros(len(batch), max_len, dtype=torch.bool)
    for i, seq in enumerate(batch):
        ids[i, :len(seq)] = seq
        mask[i, :len(seq)] = True
    return ids, mask
    ```

    </details>

**Interview takeaway:** A small loop inside collation is reasonable because sequence lengths vary; model math should remain batched.


## 17. Device and GPU Challenge


### 15 device prompts

Answer aloud or on paper before opening each answer.

**D1. Select CUDA, then MPS, then CPU.**

<details>
<summary>Show answer</summary>

Use availability checks in that priority, as in Setup.

</details>

**D2. Move model.**

<details>
<summary>Show answer</summary>

`model = model.to(device)`.

</details>

**D3. Move current batch.**

<details>
<summary>Show answer</summary>

`X, y = X.to(device), y.to(device)`.

</details>

**D4. Convert a GPU tensor requiring grad to NumPy.**

<details>
<summary>Show answer</summary>

`x.detach().cpu().numpy()`.

</details>

**D5. Check a model's device.**

<details>
<summary>Show answer</summary>

`next(model.parameters()).device` (consider parameterless modules separately).

</details>

**D6. Cause of device mismatch?**

<details>
<summary>Show answer</summary>

Operands or parameters are on different devices.

</details>

**D7. Why assign `.to(device)` back?**

<details>
<summary>Show answer</summary>

Tensor `.to` is not in-place; it can return a new tensor.

</details>

**D8. Why not move model each batch?**

<details>
<summary>Show answer</summary>

It is redundant and expensive; move it once before looping.

</details>

**D9. Should loss weights move to device?**

<details>
<summary>Show answer</summary>

Yes, tensors participating with device logits must match device.

</details>

**D10. What does `non_blocking=True` promise?**

<details>
<summary>Show answer</summary>

The copy may be asynchronous when supported, commonly pinned CPU→CUDA.

</details>

**D11. How create a tensor on x's device/dtype?**

<details>
<summary>Show answer</summary>

Use `x.new_*`, `torch.*_like(x)`, or explicit `device=x.device, dtype=x.dtype`.

</details>

**D12. Why can a plain tensor attribute stay on CPU after `model.to`?**

<details>
<summary>Show answer</summary>

Only parameters and registered buffers migrate automatically; use `register_buffer`.

</details>

**D13. Should Dataset normally emit GPU tensors from workers?**

<details>
<summary>Show answer</summary>

Usually no; emit CPU tensors and transfer batches in the main training loop.

</details>

**D14. What is MPS?**

<details>
<summary>Show answer</summary>

PyTorch's backend for Apple Silicon GPUs; feature/performance support may differ from CUDA.

</details>

**D15. How make a checkpoint portable?**

<details>
<summary>Show answer</summary>

Load with an appropriate `map_location`, then move model explicitly.

</details>


### GPU memory scenarios

Answer aloud or on paper before opening each answer.

**GPU1. Batch 32 works, 256 OOM: what do you try?**

<details>
<summary>Show answer</summary>

Reduce batch/sequence/image size, mixed precision, gradient accumulation, activation checkpointing, and inspect retained tensors.

</details>

**GPU2. Memory grows every step with `losses.append(loss)`.**

<details>
<summary>Show answer</summary>

You retain every computation graph. Store `loss.item()` or `loss.detach()` as appropriate.

</details>

**GPU3. OOM during validation despite no backward.**

<details>
<summary>Show answer</summary>

Use inference/no-grad, smaller batch, avoid retaining outputs, and ensure training graphs are released.

</details>

**GPU4. Why can `empty_cache()` fail to solve OOM?**

<details>
<summary>Show answer</summary>

It releases unused cached blocks, not live tensors; fix references/workload size.

</details>


## 18. Debugging Challenge — 25 Broken Snippets


### Bug → cause → fix

Answer aloud or on paper before opening each answer.

**Debug1. Linear expects 128 features but input is `(B,64)`.**

<details>
<summary>Show answer</summary>

Inspect the upstream shape; change preprocessing or `in_features` to match.

</details>

**Debug2. Embedding receives float token IDs.**

<details>
<summary>Show answer</summary>

Embedding indices must be integer/long: `ids.long()`.

</details>

**Debug3. CUDA model receives CPU batch.**

<details>
<summary>Show answer</summary>

Move the batch to the model's device.

</details>

**Debug4. CrossEntropy targets have shape `(B,1)`.**

<details>
<summary>Show answer</summary>

Class-index targets should normally be `(B,)` long; squeeze the singleton dimension.

</details>

**Debug5. BCE logits `(B,1)`, targets `(B,)`.**

<details>
<summary>Show answer</summary>

Make shapes identical, e.g. `targets.float().unsqueeze(1)`.

</details>

**Debug6. For logits `(B,C)`, code uses `argmax(dim=0)`.**

<details>
<summary>Show answer</summary>

Use `argmax(dim=1)` to choose a class per sample.

</details>

**Debug7. `CrossEntropyLoss(softmax(logits), y)`.**

<details>
<summary>Show answer</summary>

Pass logits directly for stability and correct semantics.

</details>

**Debug8. `BCEWithLogitsLoss(sigmoid(logits), y)`.**

<details>
<summary>Show answer</summary>

Pass raw logits directly.

</details>

**Debug9. Training loop never calls `zero_grad`.**

<details>
<summary>Show answer</summary>

Gradients accumulate across batches unintentionally.

</details>

**Debug10. Training loop never calls `backward`.**

<details>
<summary>Show answer</summary>

No gradients are computed, so optimizer step has nothing to use.

</details>

**Debug11. Training loop never calls `optimizer.step`.**

<details>
<summary>Show answer</summary>

Gradients exist but parameters never update.

</details>

**Debug12. Order is backward → zero → step.**

<details>
<summary>Show answer</summary>

Zero before forward/backward; otherwise calculated gradients are erased.

</details>

**Debug13. Validation calls `loss.backward()`.**

<details>
<summary>Show answer</summary>

Remove it and use no-grad/inference mode.

</details>

**Debug14. Validation calls `optimizer.step()`.**

<details>
<summary>Show answer</summary>

Remove it; validation must not update the model.

</details>

**Debug15. Validation omits `model.eval()`.**

<details>
<summary>Show answer</summary>

Dropout and BatchNorm retain training behavior; call eval.

</details>

**Debug16. After validation, training continues without `model.train()`.**

<details>
<summary>Show answer</summary>

Reset train mode at the start of every training epoch.

</details>

**Debug17. `history.append(loss)` every step.**

<details>
<summary>Show answer</summary>

Stores graph-connected tensors; append `loss.item()`.

</details>

**Debug18. `features = encoder(x).detach()` during fine-tuning.**

<details>
<summary>Show answer</summary>

Detach blocks encoder gradients; remove it unless intentionally frozen.

</details>

**Debug19. Parameters were set `requires_grad=False` unexpectedly.**

<details>
<summary>Show answer</summary>

Unfreeze intended parameters and rebuild/filter the optimizer if needed.

</details>

**Debug20. Optimizer was created from a discarded model instance.**

<details>
<summary>Show answer</summary>

Construct optimizer from the exact model being trained.

</details>

**Debug21. In-place ReLU modifies a tensor needed by a residual/backward path.**

<details>
<summary>Show answer</summary>

Avoid the in-place operation or preserve the required value.

</details>

**Debug22. Targets include class ID C for C classes.**

<details>
<summary>Show answer</summary>

Valid class IDs are `0..C-1`; fix label mapping/data.

</details>

**Debug23. Loss becomes NaN on first batch.**

<details>
<summary>Show answer</summary>

Check `torch.isfinite` on inputs/logits/loss and unsafe transforms such as log/division.

</details>

**Debug24. Loss explodes with learning rate 10.**

<details>
<summary>Show answer</summary>

Lower LR, inspect scaling, and consider clipping after identifying the source.

</details>

**Debug25. CNN flatten size is hard-coded incorrectly.**

<details>
<summary>Show answer</summary>

Calculate it carefully, use adaptive pooling, or infer with a dummy pass/lazy layer.

</details>


### 🔥 Must Be Able To Answer Verbally

**Your model's loss is not decreasing. Explain exactly how you would debug it.**

Write and speak your answer before opening the model response.

<details>
<summary>Show model answer</summary>

1. Verify raw data, preprocessing, labels, and label distribution.
2. Check every shape, dtype, and device.
3. Check output/target/loss compatibility and that logits—not probabilities—reach fused losses.
4. Run one batch through the model.
5. Confirm inputs, logits, and loss are finite.
6. Backpropagate once; inspect missing, zero, exploding, or non-finite gradients.
7. Snapshot a parameter and confirm `optimizer.step()` changes it.
8. Check train/eval modes and frozen parameters.
9. Try a conservative learning rate and inspect optimizer ownership.
10. Overfit 16–32 examples. If that fails, the pipeline has a fundamental bug.
11. Only then investigate capacity, regularization, normalization, and data difficulty.

</details>


## 19. Performance Challenge


### 12 performance scenarios

Answer aloud or on paper before opening each answer.

**Perf1. Low GPU utilization**

<details>
<summary>Show answer</summary>

Profile: DataLoader/preprocessing, batch too small, Python overhead, sync points, transfers, or CPU-bound ops.

</details>

**Perf2. Tiny Python loop over tensor elements**

<details>
<summary>Show answer</summary>

Vectorize with tensor operations.

</details>

**Perf3. Repeated `torch.cat` inside a loop**

<details>
<summary>Show answer</summary>

Accumulate tensors in a list and concatenate once, or preallocate.

</details>

**Perf4. Frequent `.item()` in a CUDA inner loop**

<details>
<summary>Show answer</summary>

It can force synchronization; aggregate on device and report less often.

</details>

**Perf5. Transfers CPU→GPU one sample at a time**

<details>
<summary>Show answer</summary>

Batch first, use pinned memory where useful, transfer once per batch.

</details>

**Perf6. Model spends time in input augmentation**

<details>
<summary>Show answer</summary>

Profile and parallelize/cache/optimize transforms as appropriate.

</details>

**Perf7. Mixed precision produces unstable gradients**

<details>
<summary>Show answer</summary>

Use appropriate dtype/scaling, keep sensitive ops in higher precision, inspect non-finite grads.

</details>

**Perf8. `torch.compile` makes first iteration slow**

<details>
<summary>Show answer</summary>

Compilation/warm-up cost is front-loaded; benchmark steady state and validate compatibility.

</details>

**Perf9. Excessive tiny kernels**

<details>
<summary>Show answer</summary>

Fuse/vectorize work, increase batch size where possible, consider compilation.

</details>

**Perf10. Calling `empty_cache()` every step**

<details>
<summary>Show answer</summary>

Often hurts performance and does not free live tensors; remove routine calls.

</details>

**Perf11. DataLoader with too many workers is slower**

<details>
<summary>Show answer</summary>

Worker overhead/contention can dominate; tune empirically.

</details>

**Perf12. How benchmark accelerator time?**

<details>
<summary>Show answer</summary>

Warm up and synchronize around timing, or use PyTorch profiler/events.

</details>


## 20. Transformer Shape Challenge


### 20 Transformer predictions

Answer aloud or on paper before opening each answer.

**T1. `x=(8,128,768), H=12`; head dimension?**

<details>
<summary>Show answer</summary>

`Dh=64`.

</details>

**T2. After split and transpose?**

<details>
<summary>Show answer</summary>

`(8,12,128,64)`.

</details>

**T3. QKᵀ shape?**

<details>
<summary>Show answer</summary>

`(8,12,128,128)`.

</details>

**T4. Attention weights times V?**

<details>
<summary>Show answer</summary>

`(8,12,128,64)`.

</details>

**T5. After combining heads?**

<details>
<summary>Show answer</summary>

`(8,128,768)`.

</details>

**T6. LM logits for V=50,000?**

<details>
<summary>Show answer</summary>

`(8,128,50000)`.

</details>

**T7. Token IDs before embedding?**

<details>
<summary>Show answer</summary>

`(B,T)`, normally long.

</details>

**T8. Embedding output?**

<details>
<summary>Show answer</summary>

`(B,T,D)`.

</details>

**T9. Padding mask common shape?**

<details>
<summary>Show answer</summary>

Often `(B,T)` before adapting to an API/broadcast shape.

</details>

**T10. Causal mask logical shape?**

<details>
<summary>Show answer</summary>

`(T,T)`, broadcastable over batch and heads.

</details>

**T11. Broadcast causal mask across scores?**

<details>
<summary>Show answer</summary>

`(1,1,T,T)` broadcasts over B,H.

</details>

**T12. FFN `D→4D→D` output shape?**

<details>
<summary>Show answer</summary>

`(B,T,D)`.

</details>

**T13. LayerNorm(D) changes shape?**

<details>
<summary>Show answer</summary>

No.

</details>

**T14. Residual addition requirement?**

<details>
<summary>Show answer</summary>

Both paths must have the same shape `(B,T,D)`.

</details>

**T15. CrossEntropy input after flattening LM logits?**

<details>
<summary>Show answer</summary>

`(B*T,V)`.

</details>

**T16. Flattened LM targets?**

<details>
<summary>Show answer</summary>

`(B*T,)` long.

</details>

**T17. Cross-attention Q length Tq, K/V length Tk: scores?**

<details>
<summary>Show answer</summary>

`(B,H,Tq,Tk)`.

</details>

**T18. Cross-attention output sequence length?**

<details>
<summary>Show answer</summary>

`Tq`: output `(B,H,Tq,Dh)` before combine.

</details>

**T19. KV cache per layer conceptual shape?**

<details>
<summary>Show answer</summary>

Commonly `(B,H,T_cached,Dh)` for each of K and V; exact layout varies.

</details>

**T20. Why quadratic memory in full self-attention?**

<details>
<summary>Show answer</summary>

The score/weight matrices contain `T×T` entries per batch/head.

</details>


## 21. Attention Coding Challenge


### Scaled dot-product attention from memory

Return `(output, weights)`. Boolean `mask=True` means blocked. Scale scores, mask before softmax, and multiply by V.

Predict first, then complete the next cell. Run its validation before opening the solution.


In [ ]:
def attention(Q, K, V, mask=None):
    # YOUR CODE
    return None


In [ ]:
_aq = torch.randn(2, 3, 5, 4)
_ak = torch.randn(2, 3, 5, 4)
_av = torch.randn(2, 3, 5, 6)
_causal = torch.triu(torch.ones(5, 5, dtype=torch.bool), diagonal=1)
_attn_result = attention(_aq, _ak, _av, _causal)
if _attn_result is None:
    print("PENDING: attention")
else:
    _ao, _aw = _attn_result
    assert _ao.shape == (2, 3, 5, 6) and _aw.shape == (2, 3, 5, 5)
    assert torch.allclose(_aw.sum(-1), torch.ones_like(_aw.sum(-1)), atol=1e-6)
    assert _aw.masked_select(_causal).abs().max().item() < 1e-6
    print("PASS: scaled masked attention")


<details>
    <summary>Show solution</summary>

    ```python
    def attention(Q, K, V, mask=None):
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(Q.size(-1))
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    return weights @ V, weights
    ```

    </details>

**Interview takeaway:** Scaling controls softmax saturation; mask semantics must be stated explicitly.


### Causal mask from memory

Create a boolean `T×T` mask where future positions are `True`.

Predict first, then complete the next cell. Run its validation before opening the solution.


In [ ]:
T = 6
causal_mask = None


In [ ]:
if not pending("causal mask", causal_mask):
    assert causal_mask.shape == (T, T) and causal_mask.dtype == torch.bool
    assert not causal_mask.diag().any()
    assert causal_mask[0, 1:].all() and not causal_mask[-1].any()
    print("PASS: causal mask")


<details>
<summary>Show solution</summary>

```python
causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
```

</details>


### Multi-head reshape round trip

Transform `(B,T,D) → (B,H,T,Dh) → (B,T,D)` and preserve values.

Predict first, then complete the next cell. Run its validation before opening the solution.


In [ ]:
B, T, D, H = 2, 7, 24, 4
mh_x = torch.randn(B, T, D)
mh_split = None
mh_joined = None


In [ ]:
if mh_split is None or mh_joined is None:
    print("PENDING: multi-head reshape")
else:
    assert mh_split.shape == (B, H, T, D // H)
    assert mh_joined.shape == (B, T, D)
    assert torch.equal(mh_x, mh_joined)
    print("PASS: head round trip")


<details>
    <summary>Show solution</summary>

    ```python
    mh_split = mh_x.reshape(B, T, H, D // H).transpose(1, 2)
mh_joined = mh_split.transpose(1, 2).contiguous().reshape(B, T, D)
    ```

    </details>


### Build a pre-norm Transformer block

Implement LayerNorm → batch-first MultiheadAttention → residual → LayerNorm → FFN → residual for `D=64,H=4`.

Predict first, then complete the next cell. Run its validation before opening the solution.


In [ ]:
class TransformerBlock(nn.Module):
    pass

transformer_block = None


In [ ]:
if transformer_block is None:
    print("PENDING: Transformer block")
else:
    _tb_x = torch.randn(3, 11, 64)
    assert transformer_block(_tb_x).shape == _tb_x.shape
    print("PASS: Transformer block")


<details>
    <summary>Show solution</summary>

    ```python
    class TransformerBlock(nn.Module):
    def __init__(self, d_model=64, heads=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, 4*d_model), nn.GELU(), nn.Linear(4*d_model, d_model))
    def forward(self, x):
        h = self.norm1(x)
        x = x + self.attn(h, h, h, need_weights=False)[0]
        return x + self.ffn(self.norm2(x))

transformer_block = TransformerBlock()
    ```

    </details>


### Language-model token loss

Given `(B,T,V)` logits and `(B,T)` long targets, calculate token-level CrossEntropy loss.

Predict first, then complete the next cell. Run its validation before opening the solution.


In [ ]:
_lm_logits = torch.randn(2, 5, 17)
_lm_targets = torch.randint(0, 17, (2, 5))
lm_loss = None


In [ ]:
if not pending("language-model loss", lm_loss):
    _expected = F.cross_entropy(_lm_logits.reshape(-1, 17), _lm_targets.reshape(-1))
    assert torch.allclose(lm_loss, _expected)
    print("PASS: LM loss")


<details>
<summary>Show solution</summary>

```python
lm_loss = F.cross_entropy(_lm_logits.reshape(-1, _lm_logits.size(-1)), _lm_targets.reshape(-1))
```

</details>


## 22. Code Reading Challenge


### Explain realistic snippets verbally

Answer aloud or on paper before opening each answer.

**CR1. `zero_grad(set_to_none=True)` → autocast → scaled backward/step/update**

<details>
<summary>Show answer</summary>

Mixed-precision CUDA training: clear grads, run selected ops at lower precision, scale loss to protect FP16 gradients, unscale/step safely, update scale.

</details>

**CR2. Embedding → multiply by expanded mask → sum tokens → divide by clamped valid lengths**

<details>
<summary>Show answer</summary>

Masked mean pooling.

</details>

**CR3. Conv3→16,k3,p1; MaxPool2; Conv16→32,k3,p1; MaxPool2 on `(B,3,32,32)`**

<details>
<summary>Show answer</summary>

Shapes: `(B,16,32,32)`, `(B,16,16,16)`, `(B,32,16,16)`, `(B,32,8,8)`.

</details>

**CR4. Freeze backbone; optimizer filters `requires_grad`**

<details>
<summary>Show answer</summary>

Only the unfrozen head receives gradients/optimizer state and updates.

</details>

**CR5. `register_buffer('mask', mask)`**

<details>
<summary>Show answer</summary>

Stores non-parameter module state that follows device moves and normally enters state_dict.

</details>

**CR6. `clip_grad_norm_(model.parameters(), 1.0)` after backward**

<details>
<summary>Show answer</summary>

Scales gradients when their global norm exceeds 1 before the optimizer step.

</details>

**CR7. `logits[:, :-1]` compared with `ids[:, 1:]`**

<details>
<summary>Show answer</summary>

Teacher-forced next-token alignment: each position predicts the following token.

</details>

**CR8. `torch.where(mask, x, -inf)` before softmax**

<details>
<summary>Show answer</summary>

Blocks selected logits; masked positions receive approximately zero probability.

</details>

**CR9. Snapshot state dict, train, then reload snapshot**

<details>
<summary>Show answer</summary>

Restore model weights—but clone tensors when taking an in-memory snapshot to avoid aliasing surprises.

</details>

**CR10. `model(x)` instead of `model.forward(x)`**

<details>
<summary>Show answer</summary>

The call path runs module hooks and wrapper behavior before/after `forward`.

</details>


## 23. Rapid-Fire Interview Questions


### 50 answers in ≤30 seconds each

Answer aloud or on paper before opening each answer.

**RF1. Tensor vs NumPy array?**

<details>
<summary>Show answer</summary>

Tensors support accelerators and Autograd; both provide n-dimensional array operations.

</details>

**RF2. `view()` vs `reshape()`?**

<details>
<summary>Show answer</summary>

View requires compatible strides; reshape may copy if needed.

</details>

**RF3. `cat()` vs `stack()`?**

<details>
<summary>Show answer</summary>

Cat joins an existing dimension; stack creates a new one.

</details>

**RF4. `squeeze()` vs `unsqueeze()`?**

<details>
<summary>Show answer</summary>

Remove size-1 dimensions vs insert one.

</details>

**RF5. `transpose()` vs `permute()`?**

<details>
<summary>Show answer</summary>

Transpose swaps two dims; permute reorders all listed dims.

</details>

**RF6. What is broadcasting?**

<details>
<summary>Show answer</summary>

Implicitly expanding compatible singleton/missing leading dimensions without ordinary copying.

</details>

**RF7. `max()` vs `argmax()`?**

<details>
<summary>Show answer</summary>

Maximum values versus their indices; max with a dim can return both.

</details>

**RF8. What is Autograd?**

<details>
<summary>Show answer</summary>

PyTorch's dynamic automatic-differentiation system.

</details>

**RF9. What does `requires_grad` do?**

<details>
<summary>Show answer</summary>

Requests graph tracking so gradients can propagate to the tensor.

</details>

**RF10. Why zero gradients?**

<details>
<summary>Show answer</summary>

Leaf gradients accumulate by default.

</details>

**RF11. `detach()` vs `no_grad()`?**

<details>
<summary>Show answer</summary>

Detach cuts one tensor from a graph; no-grad disables recording in a block.

</details>

**RF12. What is a leaf tensor?**

<details>
<summary>Show answer</summary>

A graph endpoint normally created by the user, where gradients accumulate.

</details>

**RF13. What is `nn.Module`?**

<details>
<summary>Show answer</summary>

Base class that registers parameters/submodules/buffers and defines model behavior.

</details>

**RF14. Why `super().__init__()`?**

<details>
<summary>Show answer</summary>

It initializes Module's registration and hook machinery.

</details>

**RF15. What does `forward()` do?**

<details>
<summary>Show answer</summary>

Defines the computation performed by the module.

</details>

**RF16. Why call `model(x)`?**

<details>
<summary>Show answer</summary>

It invokes Module's call machinery, including hooks, then forward.

</details>

**RF17. What is a logit?**

<details>
<summary>Show answer</summary>

An unnormalized model score before probability conversion.

</details>

**RF18. Why ReLU?**

<details>
<summary>Show answer</summary>

Cheap nonlinearity with useful gradients on positive inputs.

</details>

**RF19. ReLU vs GELU?**

<details>
<summary>Show answer</summary>

ReLU hard-clips negatives; GELU smoothly gates by magnitude and is common in Transformers.

</details>

**RF20. BatchNorm vs LayerNorm?**

<details>
<summary>Show answer</summary>

BatchNorm uses batch statistics per channel; LayerNorm normalizes features within each sample/token.

</details>

**RF21. What does Dropout do?**

<details>
<summary>Show answer</summary>

Randomly zeros activations during training and rescales them; disabled in eval.

</details>

**RF22. What does `model.train()` do?**

<details>
<summary>Show answer</summary>

Sets training mode recursively for behavior-sensitive modules.

</details>

**RF23. What does `model.eval()` do?**

<details>
<summary>Show answer</summary>

Sets evaluation mode recursively; it does not disable gradients.

</details>

**RF24. `eval()` vs `no_grad()`?**

<details>
<summary>Show answer</summary>

Layer behavior versus Autograd recording.

</details>

**RF25. MSE vs L1?**

<details>
<summary>Show answer</summary>

MSE penalizes large errors quadratically; L1 is more robust to outliers.

</details>

**RF26. When use BCEWithLogitsLoss?**

<details>
<summary>Show answer</summary>

Binary or independent multilabel targets with raw logits.

</details>

**RF27. When use CrossEntropyLoss?**

<details>
<summary>Show answer</summary>

Usually mutually exclusive class-index targets with class logits.

</details>

**RF28. Why no softmax before CrossEntropy?**

<details>
<summary>Show answer</summary>

The loss applies stable log-softmax internally.

</details>

**RF29. Why no sigmoid before BCEWithLogits?**

<details>
<summary>Show answer</summary>

The loss fuses sigmoid and BCE stably.

</details>

**RF30. What does an optimizer do?**

<details>
<summary>Show answer</summary>

Updates parameters from gradients according to an update rule.

</details>

**RF31. SGD vs Adam?**

<details>
<summary>Show answer</summary>

SGD uses (optionally momentum-smoothed) gradients; Adam adaptively scales using moment estimates.

</details>

**RF32. Adam vs AdamW?**

<details>
<summary>Show answer</summary>

AdamW decouples weight decay from adaptive gradient updates.

</details>

**RF33. What is weight decay?**

<details>
<summary>Show answer</summary>

A regularizing parameter shrinkage term; semantics vary by optimizer.

</details>

**RF34. What is batch size?**

<details>
<summary>Show answer</summary>

Number of samples processed per optimization batch.

</details>

**RF35. What is an epoch?**

<details>
<summary>Show answer</summary>

One pass over the training dataset.

</details>

**RF36. Dataset vs DataLoader?**

<details>
<summary>Show answer</summary>

Dataset maps indices to samples; DataLoader batches, shuffles, and loads them.

</details>

**RF37. Why shuffle training data?**

<details>
<summary>Show answer</summary>

Reduces order bias and changes minibatch composition across epochs.

</details>

**RF38. What does `num_workers` do?**

<details>
<summary>Show answer</summary>

Parallelizes data loading in subprocesses.

</details>

**RF39. What does `pin_memory` do?**

<details>
<summary>Show answer</summary>

Uses page-locked CPU memory to accelerate supported CUDA transfers.

</details>

**RF40. Why GPU OOM?**

<details>
<summary>Show answer</summary>

Live parameters, optimizer state, activations, gradients, or retained graphs exceed memory.

</details>

**RF41. What is mixed precision?**

<details>
<summary>Show answer</summary>

Using lower precision where safe to improve speed/memory while preserving stability.

</details>

**RF42. FP16 vs BF16?**

<details>
<summary>Show answer</summary>

BF16 has wider exponent range; FP16 has more mantissa precision but narrower range and often needs scaling.

</details>

**RF43. What is gradient clipping?**

<details>
<summary>Show answer</summary>

Limits gradient norm/value before an optimizer step.

</details>

**RF44. What is gradient accumulation?**

<details>
<summary>Show answer</summary>

Sum gradients across microbatches before stepping to emulate a larger batch.

</details>

**RF45. What is self-attention?**

<details>
<summary>Show answer</summary>

Each token forms a weighted mixture of values from tokens in the same sequence.

</details>

**RF46. What are Q/K/V?**

<details>
<summary>Show answer</summary>

Queries request information, keys determine compatibility, values carry mixed content.

</details>

**RF47. Why divide by `sqrt(d_k)`?**

<details>
<summary>Show answer</summary>

Controls score variance and softmax saturation.

</details>

**RF48. What is causal masking?**

<details>
<summary>Show answer</summary>

Prevents a position from attending to future positions.

</details>

**RF49. What is KV cache?**

<details>
<summary>Show answer</summary>

Stored past keys/values reused during autoregressive decoding.

</details>

**RF50. Why quadratic attention cost?**

<details>
<summary>Show answer</summary>

Full self-attention materializes/uses all `T×T` token pairs.

</details>


## 24. Mock Interview Round 1 — PyTorch Fundamentals


### 10 questions — score 0 incorrect / 1 partial / 2 correct

Answer aloud or on paper before opening each answer.

**R11. Write one line for a normal `(8,16)` tensor.**

<details>
<summary>Show answer</summary>

`torch.randn(8,16)`.

</details>

**R12. Write one line for class predictions from `(B,C)` logits.**

<details>
<summary>Show answer</summary>

`logits.argmax(dim=1)`.

</details>

**R13. Predict `stack` of three `(4,5)` tensors at dim 0.**

<details>
<summary>Show answer</summary>

`(3,4,5)`.

</details>

**R14. Predict `Linear(12,7)` on `(2,9,12)`.**

<details>
<summary>Show answer</summary>

`(2,9,7)`.

</details>

**R15. Sketch a binary classifier's final layer.**

<details>
<summary>Show answer</summary>

One raw logit per sample; no sigmoid before BCEWithLogitsLoss.

</details>

**R16. Why register layers in `__init__`?**

<details>
<summary>Show answer</summary>

So parameters are persistent, discoverable, movable, and optimizable.

</details>

**R17. Select loss for C-class classification.**

<details>
<summary>Show answer</summary>

CrossEntropyLoss with `(B,C)` logits and `(B,)` long targets.

</details>

**R18. State training-step order.**

<details>
<summary>Show answer</summary>

Zero → forward → loss → backward → step.

</details>

**R19. Loss unchanged and every grad is None: first suspects?**

<details>
<summary>Show answer</summary>

Detached graph, frozen params, no backward, wrong optimizer/model, or non-differentiable path.

</details>

**R110. Explain `eval` vs `no_grad`.**

<details>
<summary>Show answer</summary>

Eval changes module behavior; no-grad disables graph recording.

</details>


**Round 1 score: ____ / 20**


## 25. Mock Interview Round 2 — AI Engineer


### 10 harder questions — score 0 / 1 / 2

Answer aloud or on paper before opening each answer.

**R21. Design batching for variable-length token sequences.**

<details>
<summary>Show answer</summary>

Custom collate pads and returns mask; consider bucketing to reduce padding.

</details>

**R22. Make a train/eval pipeline portable across CUDA, MPS, CPU.**

<details>
<summary>Show answer</summary>

Choose device once, move model once and each batch, avoid backend-specific required code.

</details>

**R23. GPU utilization is 20%; how investigate?**

<details>
<summary>Show answer</summary>

Profile pipeline; inspect data stalls, batch size, synchronization, transfers, Python loops, tiny kernels.

</details>

**R24. Predict QKᵀ for Q/K `(8,12,128,64)`.**

<details>
<summary>Show answer</summary>

`(8,12,128,128)`.

</details>

**R25. State attention implementation in five operations.**

<details>
<summary>Show answer</summary>

QKᵀ, scale, mask, softmax, weights@V.

</details>

**R26. Parameters do not change though loss has grad_fn.**

<details>
<summary>Show answer</summary>

Check backward, non-None finite grads, optimizer ownership, zero/step order, LR, frozen params.

</details>

**R27. Sequence length doubles and OOM appears; why?**

<details>
<summary>Show answer</summary>

Full attention score memory grows roughly with T²; reduce T/batch or use memory-efficient attention/checkpointing.

</details>

**R28. How fine-tune only a head?**

<details>
<summary>Show answer</summary>

Freeze backbone, build optimizer from head/trainable params, keep mode implications in mind.

</details>

**R29. Design a small text classifier.**

<details>
<summary>Show answer</summary>

Embedding/encoder, mask-aware pooling or special token, dropout as needed, linear logits; match loss to labels.

</details>

**R210. Validation is strong offline but poor in production.**

<details>
<summary>Show answer</summary>

Investigate leakage, split strategy, drift, preprocessing skew, metric mismatch, calibration, and serving parity.

</details>


**Round 2 score: ____ / 20**


## 26. Final Capstone


### Mock Live Coding — Build and train a multiclass classifier

Use only the supplied data. In **15–30 minutes**:

1. split train/validation;
2. create DataLoaders;
3. choose an MLP;
4. select loss and optimizer;
5. write train and evaluate functions;
6. train and report accuracy;
7. keep everything device-aware.


In [ ]:
torch.manual_seed(42)

X_live = torch.randn(1000, 20)
true_w_live = torch.randn(20, 4)
y_live = (X_live @ true_w_live).argmax(dim=1)

# YOUR COMPLETE LIVE-CODING SOLUTION HERE
live_model = None
live_val_accuracy = None


In [ ]:
if live_model is None or live_val_accuracy is None:
    print("PENDING: live coding challenge")
else:
    live_model.eval()
    with torch.inference_mode():
        _live_out = live_model(X_live[:7].to(next(live_model.parameters()).device))
    assert _live_out.shape == (7, 4)
    assert isinstance(live_val_accuracy, float) and 0 <= live_val_accuracy <= 1
    assert live_val_accuracy > 0.60, "Expected meaningful accuracy on this learnable task"
    print("PASS: live-coding classifier")


<details>
<summary>Show one reference solution</summary>

```python
generator = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(TensorDataset(X_live, y_live), [800, 200], generator=generator)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128)

live_model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 4)).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(live_model.parameters(), lr=1e-2)
for epoch in range(12):
    train_one_epoch(live_model, train_loader, loss_fn, optimizer, device)
_, live_val_accuracy = evaluate(live_model, val_loader, loss_fn, device)
```

</details>


### Live-coding follow-ups

Answer aloud or on paper before opening each answer.

**Follow-up1. Why this loss?**

<details>
<summary>Show answer</summary>

One mutually exclusive class per sample: raw `(B,4)` logits and `(B,)` long IDs fit CrossEntropyLoss.

</details>

**Follow-up2. Why no softmax?**

<details>
<summary>Show answer</summary>

CrossEntropy applies stable log-softmax internally.

</details>

**Follow-up3. Why `train()`?**

<details>
<summary>Show answer</summary>

Enables training behavior for Dropout/BatchNorm.

</details>

**Follow-up4. Why `eval()`?**

<details>
<summary>Show answer</summary>

Enables deterministic evaluation behavior for those modules.

</details>

**Follow-up5. Highly imbalanced data?**

<details>
<summary>Show answer</summary>

Use appropriate split/metrics, class weighting or sampling, and inspect per-class performance.

</details>

**Follow-up6. Model overfits?**

<details>
<summary>Show answer</summary>

More data/augmentation, regularization, smaller capacity, early stopping, and verify split.

</details>

**Follow-up7. GPU OOM?**

<details>
<summary>Show answer</summary>

Smaller batch, mixed precision, accumulation, smaller inputs/model, checkpointing.

</details>

**Follow-up8. Validation stays random?**

<details>
<summary>Show answer</summary>

Check labels/split/preprocessing/modes, overfit a tiny set, inspect grads and updates.

</details>

**Follow-up9. Make training faster?**

<details>
<summary>Show answer</summary>

Profile first; then address loading, batching, precision, compilation/vectorization, and transfer overhead.

</details>

**Follow-up10. Save the model?**

<details>
<summary>Show answer</summary>

Save `state_dict` plus architecture/config and relevant optimizer/training metadata.

</details>


### Final broken AI pipeline — find approximately 15 issues


In [ ]:
# READING/DEBUGGING EXERCISE — intentionally broken; do not execute this block as a pipeline.
broken_pipeline = r'''
class BrokenDataset(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y.float()
    def __len__(self): return len(self.X) - 1
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(BrokenDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(BrokenDataset(X_val, y_val), batch_size=64, shuffle=True)

model = nn.Sequential(nn.Linear(19, 64), nn.ReLU(), nn.Linear(64, 3))
other_model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 4))
optimizer = optim.SGD(other_model.parameters(), lr=10.0)
loss_fn = nn.CrossEntropyLoss()
model.to(device)
history = []

for epoch in range(5):
    for xb, yb in train_loader:
        logits = torch.softmax(model(xb), dim=1)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.zero_grad()
        optimizer.step()
        history.append(loss)

    for xb, yb in val_loader:
        logits = model(xb.to(device))
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        pred = logits.argmax(dim=0)
'''
print(broken_pipeline)


#### Required debugging procedure

Do not randomly edit. Work in this order:

1. Inspect dataset sample.
2. Inspect batch.
3. Inspect model output shape.
4. Inspect target dtype/shape.
5. Run one forward pass.
6. Confirm finite loss.
7. Run backward.
8. Inspect gradients.
9. Confirm optimizer changes parameters.
10. Overfit 32 examples.
11. Run full training.
12. Run proper validation.

<details>
<summary>Show issue checklist and corrected direction</summary>

Issues include: wrong dataset length; float class IDs; unnecessary validation shuffle; wrong input size; wrong class count; optimizer owns another model; excessive LR; CPU batches sent to device model; softmax before CrossEntropy; missing `model.train()`; zeroing after backward; erased gradients before step; graph-connected history; no eval mode; no no-grad; validation backward and step; target device mismatch; wrong argmax dimension.

Correct the Dataset, model dimensions, optimizer ownership/LR, device movement, raw-logit loss, loop order, detached metric history, and side-effect-free validation. Then apply the assertions below to your fixed objects.

</details>


In [ ]:
def validate_fixed_pipeline(dataset, loader, model, loss_fn, optimizer, history, device):
    assert len(dataset) == len(dataset.X)
    xb, yb = next(iter(loader))
    assert xb.ndim == 2 and xb.shape[1] == 20
    assert yb.dtype == torch.long and yb.ndim == 1
    model = model.to(device)
    xb, yb = xb.to(device), yb.to(device)
    logits = model(xb)
    assert logits.shape == (len(xb), 4)
    loss = loss_fn(logits, yb)
    assert torch.isfinite(loss)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    grads = [p.grad for p in model.parameters() if p.requires_grad]
    assert grads and all(g is not None and torch.isfinite(g).all() for g in grads)
    before = [p.detach().clone() for p in model.parameters()]
    optimizer.step()
    assert any(not torch.equal(a, b) for a, b in zip(before, model.parameters()))
    assert all(isinstance(v, (int, float)) for v in history)
    snap = [p.detach().clone() for p in model.parameters()]
    model.eval()
    with torch.inference_mode():
        val_logits = model(xb)
        accuracy = (val_logits.argmax(1) == yb).float().mean().item()
    assert 0 <= accuracy <= 1
    assert all(torch.equal(a, b) for a, b in zip(snap, model.parameters()))
    return True

print("Capstone validator defined. Pass your corrected objects when ready.")


## Timed Syntax Round — 25 prompts, target 10 minutes


### Timed syntax answers

Answer aloud or on paper before opening each answer.

**TS1. Random normal `(8,16)`**

<details>
<summary>Show answer</summary>

`torch.randn(8,16)`

</details>

**TS2. Convert x to long**

<details>
<summary>Show answer</summary>

`x.long()`

</details>

**TS3. Argmax across classes**

<details>
<summary>Show answer</summary>

`logits.argmax(1)`

</details>

**TS4. Flatten except batch**

<details>
<summary>Show answer</summary>

`x.flatten(1)`

</details>

**TS5. Move x to device**

<details>
<summary>Show answer</summary>

`x = x.to(device)`

</details>

**TS6. Detach and move to CPU**

<details>
<summary>Show answer</summary>

`x.detach().cpu()`

</details>

**TS7. Linear 128→10**

<details>
<summary>Show answer</summary>

`nn.Linear(128,10)`

</details>

**TS8. LayerNorm 768**

<details>
<summary>Show answer</summary>

`nn.LayerNorm(768)`

</details>

**TS9. Embedding vocab 50k, dim 768**

<details>
<summary>Show answer</summary>

`nn.Embedding(50_000,768)`

</details>

**TS10. AdamW lr 1e-4**

<details>
<summary>Show answer</summary>

`optim.AdamW(model.parameters(), lr=1e-4)`

</details>

**TS11. Disable gradients for a block**

<details>
<summary>Show answer</summary>

`with torch.no_grad():`

</details>

**TS12. Set eval mode**

<details>
<summary>Show answer</summary>

`model.eval()`

</details>

**TS13. Set train mode**

<details>
<summary>Show answer</summary>

`model.train()`

</details>

**TS14. Zero grads efficiently**

<details>
<summary>Show answer</summary>

`optimizer.zero_grad(set_to_none=True)`

</details>

**TS15. Clip global grad norm to 1**

<details>
<summary>Show answer</summary>

`nn.utils.clip_grad_norm_(model.parameters(),1.0)`

</details>

**TS16. Concatenate a,b along dim 1**

<details>
<summary>Show answer</summary>

`torch.cat([a,b],dim=1)`

</details>

**TS17. Stack a,b on new dim 0**

<details>
<summary>Show answer</summary>

`torch.stack([a,b],dim=0)`

</details>

**TS18. NCHW→NHWC**

<details>
<summary>Show answer</summary>

`x.permute(0,2,3,1)`

</details>

**TS19. Positive mask**

<details>
<summary>Show answer</summary>

`x > 0`

</details>

**TS20. Check all finite**

<details>
<summary>Show answer</summary>

`torch.isfinite(x).all()`

</details>

**TS21. Scalar Python number**

<details>
<summary>Show answer</summary>

`x.item()`

</details>

**TS22. CrossEntropy loss object**

<details>
<summary>Show answer</summary>

`nn.CrossEntropyLoss()`

</details>

**TS23. BCE logits loss object**

<details>
<summary>Show answer</summary>

`nn.BCEWithLogitsLoss()`

</details>

**TS24. Top 5 indices by row**

<details>
<summary>Show answer</summary>

`x.topk(5,dim=1).indices`

</details>

**TS25. Identity 5×5**

<details>
<summary>Show answer</summary>

`torch.eye(5)`

</details>


## Timed Shape Round — 20 prompts, target 10 minutes


### Timed shape answers

Answer aloud or on paper before opening each answer.

**Quick1. `x=(32,3,224,224); x.flatten(1)`**

<details>
<summary>Show answer</summary>

`(32, 150528)`

</details>

**Quick2. `x=(16,20,128); x.mean(dim=1)`**

<details>
<summary>Show answer</summary>

`(16, 128)`

</details>

**Quick3. `x=(8,12,64,64); x.mean(dim=(2,3))`**

<details>
<summary>Show answer</summary>

`(8, 12)`

</details>

**Quick4. `x=(32,10); x.argmax(dim=1, keepdim=True)`**

<details>
<summary>Show answer</summary>

`(32, 1)`

</details>

**Quick5. `x=(2,3,4); x.reshape(6,4)`**

<details>
<summary>Show answer</summary>

`(6, 4)`

</details>

**Quick6. `x=(7,1,9); x.squeeze(1)`**

<details>
<summary>Show answer</summary>

`(7, 9)`

</details>

**Quick7. `x=(7,1,9); x.squeeze()`**

<details>
<summary>Show answer</summary>

`(7, 9)`

</details>

**Quick8. `x=(5,6); x.unsqueeze(0)`**

<details>
<summary>Show answer</summary>

`(1, 5, 6)`

</details>

**Quick9. `x=(5,6); x.unsqueeze(-1)`**

<details>
<summary>Show answer</summary>

`(5, 6, 1)`

</details>

**Quick10. `x=(2,3,4); x.transpose(1,2)`**

<details>
<summary>Show answer</summary>

`(2, 4, 3)`

</details>

**Quick11. `x=(8,128,768), H=12`; head dimension?**

<details>
<summary>Show answer</summary>

`Dh=64`.

</details>

**Quick12. After split and transpose?**

<details>
<summary>Show answer</summary>

`(8,12,128,64)`.

</details>

**Quick13. QKᵀ shape?**

<details>
<summary>Show answer</summary>

`(8,12,128,128)`.

</details>

**Quick14. Attention weights times V?**

<details>
<summary>Show answer</summary>

`(8,12,128,64)`.

</details>

**Quick15. After combining heads?**

<details>
<summary>Show answer</summary>

`(8,128,768)`.

</details>

**Quick16. LM logits for V=50,000?**

<details>
<summary>Show answer</summary>

`(8,128,50000)`.

</details>

**Quick17. Token IDs before embedding?**

<details>
<summary>Show answer</summary>

`(B,T)`, normally long.

</details>

**Quick18. Embedding output?**

<details>
<summary>Show answer</summary>

`(B,T,D)`.

</details>

**Quick19. Padding mask common shape?**

<details>
<summary>Show answer</summary>

Often `(B,T)` before adapting to an API/broadcast shape.

</details>

**Quick20. Causal mask logical shape?**

<details>
<summary>Show answer</summary>

`(T,T)`, broadcastable over batch and heads.

</details>


## 27. Scoring


Score representative challenge sets rather than every recall prompt.

| Area | Score |
|---|---:|
| Tensor fundamentals | /20 |
| Shape reasoning | /25 |
| Autograd | /15 |
| Models | /20 |
| Loss / optimizer | /20 |
| Training loops | /25 |
| Data pipeline | /15 |
| GPU / performance | /15 |
| Debugging | /25 |
| Transformers | /25 |
| **Total** | **/200** |

| Total | Interpretation |
|---:|---|
| 180–200 | Strong practical PyTorch readiness |
| 155–179 | Good interview readiness; review a few weak areas |
| 125–154 | Functional but several gaps remain |
| 90–124 | Needs targeted practice before interviews |
| Below 90 | Repeat core notebooks before relying on PyTorch in interviews |

This is a study benchmark, not a scientifically validated hiring score.


## 28. Weak-Area Review


| Topic | Score | Confidence | Review Notebook |
|---|---:|---|---|
| Tensors | | | 01–02 |
| Autograd | | | 03 |
| Models | | | 04 |
| Losses | | | 05 |
| Training | | | 06 |
| DataLoader | | | 07 |
| GPU | | | 08 |
| Debugging | | | 09 |
| Transformers | | | 10 |

### My PyTorch Mistake Log

| Mistake | Why I made it | Correct pattern |
|---|---|---|
| | | |
| | | |
| | | |

Record repeated mistakes immediately. A recurring error is a review target, not a character flaw.


### Interview answer practice — answer each in 30 seconds or less


### Five spoken answers

Answer aloud or on paper before opening each answer.

**Speak1. Explain `model.eval()` vs `torch.no_grad()`.**

<details>
<summary>Show answer</summary>

`eval()` changes module behavior such as Dropout and BatchNorm. `no_grad()` disables Autograd recording. Validation normally uses both.

</details>

**Speak2. Explain CrossEntropyLoss input format.**

<details>
<summary>Show answer</summary>

Pass raw class logits shaped `(B,C,...)` and usually long class indices shaped `(B,...)`; do not softmax first.

</details>

**Speak3. Why do PyTorch gradients accumulate?**

<details>
<summary>Show answer</summary>

Backward adds into leaf `.grad`, enabling multi-loss and microbatch accumulation. Zero them when starting a new update.

</details>

**Speak4. Dataset vs DataLoader?**

<details>
<summary>Show answer</summary>

Dataset defines sample access; DataLoader handles batching, order, multiprocessing, and collation.

</details>

**Speak5. Explain Q/K/V attention.**

<details>
<summary>Show answer</summary>

Queries score compatibility with keys; softmaxed scaled scores weight values, producing a context-dependent mixture.

</details>


## 29. Final Cheat Sheet — Must Know From Memory


In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 3),
        )

    def forward(self, x):
        return self.net(x)


model_pattern = Model().to(device)
loss_fn_pattern = nn.CrossEntropyLoss()
optimizer_pattern = optim.AdamW(model_pattern.parameters(), lr=1e-3)

X_pattern = torch.randn(16, 10).to(device)
y_pattern = torch.randint(0, 3, (16,)).to(device)
logits_pattern = model_pattern(X_pattern)
loss_pattern = loss_fn_pattern(logits_pattern, y_pattern)
pred_pattern = logits_pattern.argmax(dim=1)
assert logits_pattern.shape == (16, 3)


```python
# Training
model.train()
for X, y in train_loader:
    X, y = X.to(device), y.to(device)
    optimizer.zero_grad(set_to_none=True)
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()

# Validation
model.eval()
with torch.inference_mode():
    for X, y in val_loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
```

```python
class MyDataset(Dataset):
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]
```

```python
scores = (Q @ K.transpose(-2, -1)) / math.sqrt(Q.size(-1))
scores = scores.masked_fill(mask, float("-inf"))
weights = torch.softmax(scores, dim=-1)
output = weights @ V
```


### Final concept checklist

- [ ] Tensor shape reasoning, broadcasting, `cat` vs `stack`, matrix multiplication
- [ ] Autograd, accumulation, leaf tensors, `detach` vs `no_grad`
- [ ] `nn.Module`, `forward`, logits, ReLU/GELU, BatchNorm/LayerNorm
- [ ] CrossEntropyLoss, BCEWithLogitsLoss, SGD/Adam/AdamW
- [ ] Training-loop order, train vs eval, Dataset vs DataLoader
- [ ] Device handling, GPU OOM, mixed precision, gradient inspection
- [ ] Debugging non-decreasing loss and overfitting a tiny subset
- [ ] Self-attention, Q/K/V, causal masking, multi-head shapes, LM logits

### Final coding checklist

- [ ] Create, reshape, permute, mask, and multiply tensors from memory
- [ ] Write a custom module, MLP, embedding classifier, and basic CNN
- [ ] Select losses and create AdamW
- [ ] Write training and evaluation loops
- [ ] Write Dataset, DataLoader, and padding `collate_fn`
- [ ] Write device-aware, finite-value, and gradient checks
- [ ] Write scaled dot-product attention, a causal mask, and Transformer block

> Question → Recall → Write → Run → Debug → Explain → Repeat
